# Sesi 8 · Pembersihan Data dan Visualisasi

**Kursus:** Data Analysis dengan BigQuery, Python & BI
**Nama:** *tulis namamu di sini*
**Tanggal:** 26 November 2026

Data mentah hampir tidak pernah langsung siap dianalisis. Hari ini kita:

1. **Membersihkan** tabel `orders`: mengecek nilai kosong, duplikat, tipe data, dan logika.
2. **Membuat 3 grafik** yang menyampaikan satu pesan masing-masing: tren pendapatan bulanan, komposisi kategori, dan sebaran umur pelanggan.
3. **Menyimpan grafik** sebagai gambar untuk portofolio.

> Simpan salinan dulu: **File → Save a copy in Drive**.

## 0. Persiapan

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from google.cloud import bigquery

PROJECT_ID = "isi-project-id-kamu"   # <-- GANTI dengan Project ID-mu
client = bigquery.Client(project=PROJECT_ID)
pd.set_option("display.float_format", "{:,.2f}".format)

def jalankan_query(sql):
    """Menjalankan query SQL di BigQuery dan mengembalikan hasilnya sebagai DataFrame."""
    return client.query(sql).to_dataframe()

# Bagian A · Membersihkan data pesanan

Kita ambil tabel `orders` **apa adanya**, tanpa filter, supaya bisa melihat semua masalahnya.

In [ ]:
orders = jalankan_query("""
SELECT order_id, user_id, status, gender, created_at,
       shipped_at, delivered_at, returned_at, num_of_item
FROM `bigquery-public-data.thelook_ecommerce.orders`
""")
print(orders.shape)
orders.head()

## A1. Checklist pembersihan

Setiap kali menerima data baru, cek lima hal ini:

| # | Cek | Perintah pandas |
| --- | --- | --- |
| 1 | Nilai kosong | `df.isna().sum()` |
| 2 | Duplikat | `df.duplicated().sum()`, `df["id"].duplicated().sum()` |
| 3 | Tipe data | `df.dtypes` |
| 4 | Nilai kategori yang aneh | `df["kolom"].value_counts()` |
| 5 | Logika bisnis | contoh: tanggal diterima tidak boleh sebelum tanggal dikirim |

Lalu **tulis keputusanmu**: apa yang diubah, apa yang dibiarkan, dan kenapa.

### 1. Nilai kosong

In [ ]:
orders.isna().sum()

Banyak `shipped_at`, `delivered_at`, dan `returned_at` yang kosong. **Apakah ini masalah?** Belum tentu. Cek kaitannya dengan status:

In [ ]:
# Untuk setiap status: berapa persen shipped_at, delivered_at, dan returned_at yang kosong?
orders.groupby("status")[["shipped_at", "delivered_at", "returned_at"]].agg(lambda kolom: kolom.isna().mean() * 100).round(1)

Pesanan `Processing` dan `Cancelled` memang belum/tidak dikirim, jadi `shipped_at` kosong itu **wajar**. Ini disebut *missing yang bermakna*: kosong karena kejadiannya belum terjadi.

**Keputusan:** jangan diisi (`fillna`) dan jangan dibuang. Mengisi tanggal pengiriman palsu justru membuat data salah.

### 2. Duplikat

In [ ]:
print("Baris duplikat persis :", orders.duplicated().sum())
print("order_id duplikat     :", orders["order_id"].duplicated().sum())

Kalau ada duplikat, buang dengan `orders = orders.drop_duplicates()` atau `orders.drop_duplicates(subset="order_id")`.

### 3. Tipe data

In [ ]:
orders.dtypes

Kolom waktu dari BigQuery sudah bertipe `datetime64[..., UTC]`, artinya waktu dengan zona UTC. Kita buang info zona waktunya supaya lebih mudah diolah (nilainya tetap UTC). `pd.to_datetime()` juga mengubah kolom teks menjadi tanggal kalau tipenya belum benar.

In [ ]:
kolom_waktu = ["created_at", "shipped_at", "delivered_at", "returned_at"]
for kolom in kolom_waktu:
    orders[kolom] = pd.to_datetime(orders[kolom], utc=True).dt.tz_localize(None)

orders[kolom_waktu].dtypes

### 4. Nilai kategori

In [ ]:
print(orders["status"].value_counts(), "\n")
print(orders["gender"].value_counts())

Kalau ada nilai seperti `complete` dan `Complete`, atau `Complete ` (ada spasi), seragamkan dengan `.str.strip().str.title()`.

### 5. Logika bisnis

In [ ]:
# Pesanan yang diterima SEBELUM dikirim? Seharusnya tidak ada.
salah_urutan = orders[orders["delivered_at"] < orders["shipped_at"]]
print("Diterima sebelum dikirim:", len(salah_urutan))

# Jumlah barang harus minimal 1
print("num_of_item < 1        :", (orders["num_of_item"] < 1).sum())

# Ada pesanan dari masa depan?
print("created_at > hari ini  :", (orders["created_at"] > pd.Timestamp.now()).sum())

## A2. Menambah kolom yang berguna

In [ ]:
orders["order_date"] = orders["created_at"].dt.normalize()                     # tanggal tanpa jam
orders["order_month"] = orders["created_at"].dt.to_period("M").dt.to_timestamp()  # awal bulan
orders["hari_sampai"] = (orders["delivered_at"] - orders["created_at"]).dt.days    # lama pengiriman

orders[["created_at", "order_date", "order_month", "hari_sampai"]].head()

In [ ]:
orders["hari_sampai"].describe()

## A3. Membuang bulan berjalan

Bulan ini belum selesai, jadi angkanya pasti lebih kecil. Kalau ikut digrafikkan, tren terlihat "anjlok" di ujung.

In [ ]:
awal_bulan_ini = pd.Timestamp.today().normalize().replace(day=1)
print("Data dipakai sampai sebelum:", awal_bulan_ini.date())

orders_bersih = orders[orders["created_at"] < awal_bulan_ini].copy()
print("Sebelum:", len(orders), "baris | Sesudah:", len(orders_bersih), "baris")

## A4. Catatan pembersihan

Tulis keputusan pembersihanmu, supaya orang lain (dan kamu sendiri bulan depan) tahu apa yang terjadi pada data:

| Temuan | Keputusan | Alasan |
| --- | --- | --- |
| `shipped_at`/`delivered_at` kosong untuk Processing & Cancelled | Dibiarkan | Kosong karena memang belum terjadi |
| Kolom waktu bertipe UTC | Info zona waktu dibuang, nilai tetap UTC | Memudahkan pengolahan |
| Bulan berjalan belum lengkap | Dibuang | Supaya tren tidak menyesatkan |
| *(tambahkan temuanmu)* | | |

# Bagian B · Visualisasi

## B1. Prinsip grafik yang baik

1. **Satu grafik, satu pesan.** Tentukan dulu pesannya, baru pilih grafiknya.
2. **Judul = kesimpulan.** "Jeans menyumbang 12% pendapatan" lebih berguna daripada "Pendapatan per kategori".
3. **Pilih bentuk sesuai tugasnya:**

| Ingin menunjukkan | Grafik |
| --- | --- |
| Perubahan dari waktu ke waktu | Garis (*line*) |
| Membandingkan kategori | Batang horizontal, **diurutkan** |
| Sebaran angka (umur, harga) | Histogram, boxplot |
| Hubungan dua angka | *Scatter* |

4. **Kurangi hiasan.** Tanpa 3D, tanpa bayangan, tanpa warna-warni yang tidak berarti. Pakai satu warna, lalu satu warna sorotan untuk hal yang ingin ditekankan.
5. **Beri label** sumbu dan satuan (USD, %, orang).
6. **Satu sumbu Y saja.** Dua ukuran dengan skala berbeda → buat dua grafik.

In [ ]:
# Gaya grafik untuk seluruh notebook
WARNA_UTAMA = "#2a78d6"    # biru: warna data
WARNA_SOROT = "#eb6834"    # oranye: untuk menyorot satu hal penting
WARNA_TEKS = "#5f6368"     # abu-abu: teks pendukung

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 100,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e8eaed",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.edgecolor": "#bdc1c6",
    "xtick.color": WARNA_TEKS,
    "ytick.color": WARNA_TEKS,
})

os.makedirs("images", exist_ok=True)   # folder untuk menyimpan gambar grafik

Untuk grafik pendapatan dan kategori kita pakai **tabel analisis** dari sesi 7 (satu baris per barang terjual).

In [ ]:
SQL_TABEL_ANALISIS = """
WITH item_valid AS (
  SELECT id, order_id, user_id, product_id, status, created_at, sale_price
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
    AND DATE(created_at) < DATE_TRUNC(CURRENT_DATE(), MONTH)
)
SELECT
  i.id AS order_item_id, i.order_id, i.user_id, i.product_id, i.status,
  DATE(i.created_at) AS order_date,
  DATE_TRUNC(DATE(i.created_at), MONTH) AS order_month,
  i.sale_price, p.cost, i.sale_price - p.cost AS margin,
  p.category, COALESCE(p.brand, 'Tidak diketahui') AS brand, p.department,
  u.age,
  CASE
    WHEN u.age < 18 THEN '1. Di bawah 18'
    WHEN u.age < 25 THEN '2. 18-24'
    WHEN u.age < 35 THEN '3. 25-34'
    WHEN u.age < 45 THEN '4. 35-44'
    WHEN u.age < 55 THEN '5. 45-54'
    ELSE '6. 55+'
  END AS age_band,
  u.gender, u.country, u.traffic_source
FROM item_valid AS i
JOIN `bigquery-public-data.thelook_ecommerce.products` AS p ON i.product_id = p.id
JOIN `bigquery-public-data.thelook_ecommerce.users` AS u ON i.user_id = u.id
"""
penjualan = jalankan_query(SQL_TABEL_ANALISIS)
penjualan["order_month"] = pd.to_datetime(penjualan["order_month"])
penjualan.shape

## B2. Grafik 1: tren pendapatan bulanan (garis)

**Pertanyaan:** apakah bisnis kita tumbuh?

Kita pakai 24 bulan terakhir supaya tren terbaru terlihat jelas, lalu menghitung pertumbuhan 12 bulan terakhir untuk judulnya.

In [ ]:
bulanan = penjualan.groupby("order_month")["sale_price"].sum()
bulanan_24 = bulanan.tail(24)

# Pesan untuk judul: bandingkan 12 bulan terakhir dengan 12 bulan sebelumnya
pendapatan_12_terakhir = bulanan.tail(12).sum()
pendapatan_12_sebelumnya = bulanan.tail(24).head(12).sum()
pertumbuhan = (pendapatan_12_terakhir / pendapatan_12_sebelumnya - 1) * 100
arah = "naik" if pertumbuhan >= 0 else "turun"
print(f"Pertumbuhan 12 bulan terakhir: {pertumbuhan:.0f}%")

In [ ]:
fig, ax = plt.subplots()
ax.plot(bulanan_24.index, bulanan_24.values, color=WARNA_UTAMA, linewidth=2)

# Sorot bulan terakhir dengan titik dan label angka
ax.scatter(bulanan_24.index[-1], bulanan_24.values[-1], color=WARNA_UTAMA, s=50, zorder=3)
ax.annotate(f"{bulanan_24.values[-1]:,.0f} USD",
            (bulanan_24.index[-1], bulanan_24.values[-1]),
            textcoords="offset points", xytext=(-10, 10), ha="right", color=WARNA_TEKS)

ax.set_title(f"Pendapatan 12 bulan terakhir {arah} {abs(pertumbuhan):.0f}% dibanding tahun sebelumnya")
ax.set_ylabel("Pendapatan (USD)")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.set_ylim(bottom=0)   # sumbu Y mulai dari 0 supaya perubahan tidak dilebih-lebihkan
fig.text(0.01, -0.02, "Sumber: TheLook (BigQuery). Tidak termasuk barang Cancelled/Returned.",
         color=WARNA_TEKS, fontsize=9)
plt.tight_layout()
plt.savefig("images/01_pendapatan_bulanan.png", dpi=150, bbox_inches="tight")
plt.show()

## B3. Grafik 2: komposisi pendapatan per kategori (batang horizontal)

**Pertanyaan:** kategori mana yang menopang bisnis?

Batang **horizontal** memudahkan membaca nama kategori yang panjang. **Urutkan** dari terbesar supaya mata langsung menemukan yang penting.

In [ ]:
per_kategori = penjualan.groupby("category")["sale_price"].sum().sort_values(ascending=False)
porsi = per_kategori / per_kategori.sum() * 100
top10 = porsi.head(10).sort_values()   # diurutkan naik karena barh menggambar dari bawah

kategori_teratas = porsi.index[0]
print(porsi.head(10).round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
warna = [WARNA_SOROT if k == kategori_teratas else WARNA_UTAMA for k in top10.index]
ax.barh(top10.index, top10.values, color=warna, height=0.7)

# Label persentase di ujung setiap batang
for y, nilai in enumerate(top10.values):
    ax.text(nilai + 0.2, y, f"{nilai:.1f}%", va="center", color=WARNA_TEKS, fontsize=10)

ax.set_title(f"{kategori_teratas} menyumbang {porsi.iloc[0]:.0f}% pendapatan, terbesar dari semua kategori")
ax.set_xlabel("Porsi pendapatan (%)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.savefig("images/02_porsi_kategori.png", dpi=150, bbox_inches="tight")
plt.show()

## B4. Grafik 3: sebaran umur pelanggan (histogram)

**Pertanyaan:** berapa umur pelanggan kita?

Histogram membagi umur ke dalam "wadah" (*bin*) dan menghitung berapa pelanggan di setiap wadah.

In [ ]:
users = jalankan_query("""
SELECT id, age, gender, country, traffic_source
FROM `bigquery-public-data.thelook_ecommerce.users`
""")
users["age"].describe()

In [ ]:
umur_median = users["age"].median()
porsi_18_45 = users["age"].between(18, 45).mean() * 100

fig, ax = plt.subplots()
ax.hist(users["age"], bins=range(10, 76, 5), color=WARNA_UTAMA, edgecolor="white", linewidth=2)
ax.axvline(umur_median, color=WARNA_SOROT, linewidth=2, linestyle="--")
ax.text(umur_median + 1, ax.get_ylim()[1] * 0.95, f"Median: {umur_median:.0f} tahun",
        color=WARNA_TEKS, va="top")

ax.set_title(f"{porsi_18_45:.0f}% pelanggan berumur 18-45 tahun; umurnya menyebar cukup merata")
ax.set_xlabel("Umur (tahun)")
ax.set_ylabel("Jumlah pelanggan")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.savefig("images/03_sebaran_umur.png", dpi=150, bbox_inches="tight")
plt.show()

> **Cek dulu judulnya.** Judul di atas dibuat dari angka yang dihitung, tapi kalimat "menyebar cukup merata" adalah penilaian. Lihat grafiknya: apakah benar? Kalau tidak, ganti judulnya. Judul harus sesuai dengan apa yang terlihat.

## B5. Menyimpan grafik untuk portofolio

Gambar tersimpan di folder `images/` di Colab (klik ikon folder di panel kiri). File di Colab **hilang** saat sesi ditutup, jadi unduh sekarang:

In [ ]:
from google.colab import files

for nama_file in sorted(os.listdir("images")):
    files.download(f"images/{nama_file}")

Lalu di GitHub: buat folder `images` (**Add file → Create new file** → `images/README.md`), masuk ke folder itu, dan unggah ketiga gambar.

Gambar bisa ditampilkan di README dengan:

```markdown
![Pendapatan bulanan](images/01_pendapatan_bulanan.png)
```

## Giliran kamu (±10 menit)

**G1.** Buat grafik batang horizontal **jumlah pesanan per status** dari `orders_bersih`, diurutkan dari terbanyak. Tulis judul yang berisi kesimpulan.

In [ ]:
# G1

**G2.** Dari `orders_bersih`, berapa rata-rata `hari_sampai` per bulan dalam 12 bulan terakhir? Apakah pengiriman makin cepat atau makin lambat? Buat grafik garisnya.

In [ ]:
# G2

# PR sesi 8

Kerjakan, jalankan semua sel, lalu unggah notebook ini ke `notebooks/08_cleaning_and_charts.ipynb` dan gambar-gambarnya ke folder `images/`.

## PR1. Outlier harga di tabel products

**Outlier** = nilai yang jauh berbeda dari kebanyakan data. Cara umum menemukannya adalah **metode IQR**:

- IQR = Q3 − Q1 (rentang 50% data di tengah).
- Batas bawah = Q1 − 1,5 × IQR; batas atas = Q3 + 1,5 × IQR.
- Nilai di luar batas = outlier.

Outlier **tidak otomatis dibuang**. Tanyakan: apakah ini kesalahan data, atau memang produk premium yang sah?

In [ ]:
products = jalankan_query("""
SELECT id, name, brand, category, department, cost, retail_price
FROM `bigquery-public-data.thelook_ecommerce.products`
""")
products["retail_price"].describe()

In [ ]:
# Boxplot: titik-titik di luar "kumis" adalah outlier
fig, ax = plt.subplots(figsize=(4, 6))
ax.boxplot(products["retail_price"], widths=0.5)
ax.set_title("Sebaran harga produk")
ax.set_ylabel("Harga (USD)")
ax.set_xticks([])
plt.show()

In [ ]:
# PR1a. Hitung Q1, Q3, IQR, batas bawah, dan batas atas retail_price
q1 = products["retail_price"].quantile(0.25)
q3 = products["retail_price"].quantile(0.75)
# TODO: lanjutkan

In [ ]:
# PR1b. Berapa produk yang termasuk outlier? Kategori apa saja yang paling banyak?
# Petunjuk: products[products["retail_price"] > batas_atas]["category"].value_counts()

In [ ]:
# PR1c. Cek juga kolom name dan brand: ada yang kosong? Isi brand yang kosong
#       dengan "Tidak diketahui" menggunakan .fillna()

**Keputusanmu tentang outlier:** dibuang, dibiarkan, atau dianalisis terpisah? Kenapa? *(tulis di sini)*

## PR2. Tiga grafik, masing-masing dengan satu insight

Buat **3 grafik** baru (bebas topiknya, dari `penjualan`, `orders_bersih`, `users`, atau `products`). Untuk setiap grafik:

- Judul berupa kesimpulan.
- Simpan sebagai PNG di `images/` (nama file jelas, misalnya `04_pendapatan_per_negara.png`).
- Tulis **satu kalimat insight** di sel teks di bawahnya.

Ide: pendapatan per negara, margin per kategori, AOV per `traffic_source`, porsi pesanan Returned per bulan, perbandingan pria vs wanita per kategori.

In [ ]:
# PR2 - grafik 1

*Insight grafik 1:*

In [ ]:
# PR2 - grafik 2

*Insight grafik 2:*

In [ ]:
# PR2 - grafik 3

*Insight grafik 3:*